# LOGISCALE BIGDATA — WEEK 2
# Big Data Processing & Advanced Analytics using PySpark

## SECTION 1: SPARK SESSION & DATA LOADING

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import logging

logging.basicConfig(level=logging.INFO)
log = logging.getLogger("week2")

spark = SparkSession.builder \
    .appName("LogiScale-Week2-BigDataProcessing") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

CSV_PATH = r"C:\Users\Pravith Kumar J\Downloads\final_output.csv"

log.info(f"Loading clean CSV: {CSV_PATH}")

df = spark.read.csv(CSV_PATH, header=True, inferSchema=True)
df.cache()

log.info(f"Rows loaded: {df.count():,}")

INFO:numexpr.utils:NumExpr defaulting to 12 threads.
INFO:week2:Loading clean CSV: C:\Users\Pravith Kumar J\Downloads\final_output.csv
INFO:week2:Rows loaded: 172,765


## SECTION 2: ROUTE EFFICIENCY

In [2]:
route_efficiency = df.groupBy(
    "Order Region", "Shipping Mode", "Market"
).agg(
    F.count("*").alias("total_shipments"),
    F.round(F.avg("delay_days"), 2).alias("avg_delay_days"),
    F.max("delay_days").alias("max_delay_days"),
    F.count(F.when(F.col("delay_days") > 0, 1)).alias("late_shipments"),
    F.round(
        F.count(F.when(F.col("delay_days") > 0, 1)) / F.count("*") * 100, 1
    ).alias("late_pct"),
    F.round(F.sum("sales"), 2).alias("total_sales"),
    F.round(F.avg("order_profit"), 2).alias("avg_profit")
).orderBy(F.desc("late_pct"))

route_efficiency.show(10, truncate=False)

+---------------+-------------+------------+---------------+--------------+--------------+--------------+--------+-----------+----------+
|Order Region   |Shipping Mode|Market      |total_shipments|avg_delay_days|max_delay_days|late_shipments|late_pct|total_sales|avg_profit|
+---------------+-------------+------------+---------------+--------------+--------------+--------------+--------+-----------+----------+
|Caribbean      |First Class  |LATAM       |1132           |1.0           |1             |1132          |100.0   |221960.57  |25.25     |
|South America  |First Class  |LATAM       |2154           |1.0           |1             |2154          |100.0   |425869.14  |23.76     |
|Central America|First Class  |LATAM       |4185           |1.0           |1             |4185          |100.0   |844222.57  |22.1      |
|Southeast Asia |First Class  |Pacific Asia|1482           |1.0           |1             |1482          |100.0   |299875.42  |21.41     |
|South Asia     |First Class  |Pac

## SECTION 3: SHIPPING MODE EFFICIENCY

In [3]:
shipping_mode_eff = df.groupBy("Shipping Mode").agg(
    F.count("*").alias("shipments"),
    F.round(F.avg("Days for shipping (real)"), 2).alias("avg_real_days"),
    F.round(F.avg("Days for shipment (scheduled)"), 2).alias("avg_sched_days"),
    F.round(F.avg("delay_days"), 2).alias("avg_delay"),
    F.count(F.when(F.col("late_risk") == 1, 1)).alias("at_risk"),
    F.round(
        F.count(F.when(F.col("late_risk") == 1, 1)) / F.count("*") * 100, 1
    ).alias("risk_pct"),
    F.round(F.sum("sales"), 2).alias("total_sales"),
    F.round(F.avg("order_profit"), 2).alias("avg_profit")
).orderBy("avg_delay")

shipping_mode_eff.show(truncate=False)

+--------------+---------+-------------+--------------+---------+-------+--------+-------------+----------+
|Shipping Mode |shipments|avg_real_days|avg_sched_days|avg_delay|at_risk|risk_pct|total_sales  |avg_profit|
+--------------+---------+-------------+--------------+---------+-------+--------+-------------+----------+
|Standard Class|103153   |3.99         |4.0           |-0.01    |41023  |39.8    |2.109082732E7|22.03     |
|Same Day      |9293     |0.48         |0.0           |0.48     |4454   |47.9    |1854872.21   |20.9      |
|First Class   |26513    |2.0          |1.0           |1.0      |26513  |100.0   |5408068.56   |23.26     |
|Second Class  |33806    |3.99         |2.0           |1.99     |26987  |79.8    |6860661.56   |21.38     |
+--------------+---------+-------------+--------------+---------+-------+--------+-------------+----------+



## SECTION 4: WINDOW FUNCTIONS

In [4]:
monthly = df.groupBy("year", "month").agg(
    F.count("Order Id").alias("orders"),
    F.sum("quantity").alias("units_sold"),
    F.sum("sales").alias("monthly_revenue"),
    F.sum("order_profit").alias("monthly_profit"),
    F.avg("delay_days").alias("avg_delay")
).orderBy("year", "month")

window_time = Window.orderBy("year", "month")

monthly_windowed = monthly \
    .withColumn("running_revenue", F.sum("monthly_revenue").over(window_time)) \
    .withColumn("running_profit", F.sum("monthly_profit").over(window_time))

monthly_windowed.show(10)

+----+-----+------+----------+------------------+------------------+------------------+------------------+------------------+
|year|month|orders|units_sold|   monthly_revenue|    monthly_profit|         avg_delay|   running_revenue|    running_profit|
+----+-----+------+----------+------------------+------------------+------------------+------------------+------------------+
|2015|    1|  5114|     11350| 1008854.429428168|105773.57018058503|0.5506452874462261| 1008854.429428168|105773.57018058503|
|2015|    2|  4487|      9912| 881666.9872085945| 93009.96017430202| 0.594383775351014|1890521.4166367624|198783.53035488704|
|2015|    3|  5127|     11526|1007758.0496462269|  107401.920248345|0.5724595279890774|2898279.4662829894|  306185.450603232|
|2015|    4|  4915|     10790| 972303.1490619674|102071.90997068997|0.5424211597151577|3870582.6153449565|  408257.360573922|
|2015|    5|  5087|     11314| 996912.9091998739|108951.47009688399|0.5553371338706506|  4867495.52454483|  517208.830

## SECTION 5: ANOMALY DETECTION

In [5]:
window_region = Window.partitionBy("Order Region")

df_anomaly = df \
    .withColumn("region_avg_delay", F.avg("delay_days").over(window_region)) \
    .withColumn("region_stddev", F.stddev("delay_days").over(window_region)) \
    .withColumn(
        "z_score",
        (F.col("delay_days") - F.col("region_avg_delay")) / F.col("region_stddev")
    )

anomalies = df_anomaly.filter(F.abs(F.col("z_score")) > 2)

anomalies.show(5)

+--------+------------------------+-----------------------------+-----------------+------------------+---------------+------------------+-----------+----------------+-----------------+----------------+-----------+----------------+--------------+----------------+-------------+---------------+-----------+------------+------+-----------+-------------+-----------------+-----------------------+--------+----------------------+-------------------+------------------------+-------------+------------------------+-----------------------+-------------------+-----------+----------------+----------------------+------------+-----------+------------+-------------+---------------+-------------------+-------------------+--------------------+--------------------+-------------+--------------+--------------------------+-------------+----------+-------------+----------+------------+-------------+--------+------------+---------+----+-----+-------+------------+-------------------+------------------+----------

## SECTION 6: PROFIT ANALYSIS

In [6]:
profit_analysis = df.groupBy(
    "Customer Segment", "Category Name", "Market"
).agg(
    F.count("*").alias("orders"),
    F.sum("sales").alias("total_sales"),
    F.sum("order_profit").alias("total_profit"),
    F.avg("profit_ratio").alias("avg_margin")
).orderBy(F.desc("total_profit"))

profit_analysis.show(10)

+----------------+----------------+------------+------+-----------------+------------------+-------------------+
|Customer Segment|   Category Name|      Market|orders|      total_sales|      total_profit|         avg_margin|
+----------------+----------------+------------+------+-----------------+------------------+-------------------+
|        Consumer|         Fishing|      Europe|  2420|967951.6266200074|  111030.041162662|0.12807438020123965|
|        Consumer|         Fishing|       LATAM|  2481|992350.4072910079|107718.27120843099|0.12054010489600972|
|        Consumer|          Cleats|       LATAM|  3594|647172.1378549604|   75906.479879858|0.12654980522481912|
|       Corporate|         Fishing|       LATAM|  1529|611569.4368190033| 71939.39096522095|0.13084368872465674|
|        Consumer|         Fishing|        USCA|  1332|532773.3746520068| 71563.19045584396|0.14894894901201208|
|        Consumer|         Fishing|Pacific Asia|  1839|735563.2402290057| 70350.16067484501|0.10

## SECTION 7: SAVE OUTPUTS

In [7]:
route_efficiency.toPandas().to_csv("agg_route_efficiency.csv", index=False)
shipping_mode_eff.toPandas().to_csv("agg_shipping_mode.csv", index=False)
monthly_windowed.toPandas().to_csv("agg_monthly_trends.csv", index=False)
profit_analysis.toPandas().to_csv("agg_profit_margin.csv", index=False)
anomalies.toPandas().to_csv("agg_anomalies.csv", index=False)

## SECTION 7: SAVE OUTPUTS

In [8]:
print("\n" + "="*55)
print("  WEEK 2 COMPLETE — SUMMARY")
print("="*55)
print(f"  Rows processed         : {df.count():,}")
print(f"  Regions analysed       : {route_efficiency.count()}")
print(f"  Monthly records        : {monthly_windowed.count()}")
print(f"  Anomalies detected     : {anomalies.count()}")
print("="*55)

spark.stop()


  WEEK 2 COMPLETE — SUMMARY
  Rows processed         : 172,765
  Regions analysed       : 92
  Monthly records        : 37
  Anomalies detected     : 6697
